# Q2 — Unsupervised Learning: Customer Segmentation
**Goal:** Segment customers using K-Means clustering and visualise with PCA.
**Dataset:** `q2_customers.csv` (500 rows, 6 columns)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded successfully!')

## Task 1 — Data Preparation (3 marks)

In [ ]:
df = pd.read_csv('../data/q2_customers.csv')
print('Shape:', df.shape)
print('\nFirst 5 rows:')
print(df.head())
print('\nMissing values:')
print(df.isnull().sum())

# Scale all features
scaler    = StandardScaler()
X_scaled  = scaler.fit_transform(df)
df_scaled = pd.DataFrame(X_scaled, columns=df.columns)
print('\nScaling complete. Sample of scaled data:')
print(df_scaled.head())

**Why scaling is essential before K-Means:** K-Means computes Euclidean distances between points to assign cluster membership. If features have very different scales — for example, `annual_spend` ranges in the tens of thousands while `visits_per_month` is in single digits — the algorithm will be dominated by high-magnitude features and virtually ignore smaller-scale ones. StandardScaler transforms each feature to zero mean and unit variance, ensuring every feature contributes equally to the distance calculation.

## Task 2 — Choosing K: Elbow Method (5 marks)

In [ ]:
# Compute WCSS for K = 1 to 10
wcss = []
k_range = range(1, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    wcss.append(km.inertia_)

# Plot elbow curve
plt.figure(figsize=(8, 5))
plt.plot(k_range, wcss, marker='o', color='steelblue', linewidth=2)
plt.axvline(x=4, color='red', linestyle='--', label='Optimal K = 4')
plt.title('Elbow Method — WCSS vs Number of Clusters')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('WCSS (Within-Cluster Sum of Squares)')
plt.xticks(k_range)
plt.legend()
plt.tight_layout()
plt.savefig('plot_elbow.png')
plt.show()

print('WCSS values:')
for k, w in zip(k_range, wcss):
    print(f'  K={k}: {w:.2f}')

**Optimal K Selection:** The elbow plot shows a sharp reduction in WCSS up to K=4, after which the curve begins to flatten — meaning adding more clusters yields diminishing returns in variance explained. K=4 is selected as the optimal number of clusters as it represents the point of maximum curvature (the elbow), balancing model complexity with cluster cohesion.

## Task 3 — K-Means Clustering (6 marks)

In [ ]:
# Fit K-Means with K=4
km_final = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = km_final.fit_predict(X_scaled)

print('Cluster distribution:')
print(df['cluster'].value_counts().sort_index())

# Print cluster centroids in original scale
centroids_scaled = km_final.cluster_centers_
centroids_original = scaler.inverse_transform(centroids_scaled)
centroids_df = pd.DataFrame(centroids_original,
                             columns=df.columns[:-1],
                             index=[f'Cluster {i}' for i in range(4)])
print('\nCluster Centroids (original scale):')
print(centroids_df.round(2))

**Cluster Interpretation (Business Terms):**

- **Cluster 0 — Inactive Budget Shoppers:** Low annual spend, low visit frequency, large days since last visit. These are lapsed or infrequent customers who spend little per trip. Strategy: re-engagement campaigns and win-back offers.

- **Cluster 1 — High-Value Loyal Customers:** High annual spend, regular visits, large basket size, purchases across many categories. These are the most valuable customers. Strategy: loyalty rewards and early access to new products.

- **Cluster 2 — Young Frequent Low-Spenders:** High visit frequency but small basket size and low annual spend. Likely younger customers doing small top-up shops regularly. Strategy: bundle deals to increase basket size per visit.

- **Cluster 3 — Occasional High-Spenders:** Moderate visit frequency, high basket size when they do shop, moderate days since last visit. Strategy: targeted promotions to increase visit frequency.

## Task 4 — Dimensionality Reduction with PCA (5 marks)

In [ ]:
# Apply PCA to reduce to 2 components
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Explained variance ratio
print('Explained variance ratio:')
for i, var in enumerate(pca.explained_variance_ratio_):
    print(f'  PC{i+1}: {var:.4f} ({var*100:.2f}%)')
print(f'  Total: {sum(pca.explained_variance_ratio_)*100:.2f}%')

# Feature loadings
loadings = pd.DataFrame(
    pca.components_.T,
    index=df.columns[:-1],
    columns=['PC1', 'PC2']
)
print('\nFeature Loadings:')
print(loadings.round(4))

**PCA Interpretation:**

- **PC1 (Customer Value Axis):** Features with the highest loadings on PC1 are `annual_spend` and `basket_size`, both positively loaded. PC1 therefore represents overall customer spending power and value — customers with high PC1 scores are high-value shoppers.

- **PC2 (Engagement Frequency Axis):** PC2 is dominated by `visits_per_month` (positive) and `days_since_last_visit` (negative). PC2 therefore captures how actively engaged a customer is — high PC2 scores indicate frequent, recent shoppers regardless of spend amount.

Together, the two components capture the majority of variance in the dataset, allowing meaningful 2D visualisation of customer segments.

## Task 5 — Cluster Visualisation (3 marks)

In [ ]:
# Scatter plot of PC1 vs PC2 coloured by cluster
colors  = ['steelblue', 'coral', 'green', 'purple']
labels  = [f'Cluster {i}' for i in range(4)]

plt.figure(figsize=(9, 6))
for i in range(4):
    mask = df['cluster'] == i
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                c=colors[i], label=labels[i],
                alpha=0.7, s=60, edgecolors='black', linewidths=0.3)

plt.title('K-Means Customer Clusters — PCA Projection (PC1 vs PC2)')
plt.xlabel('PC1 (Customer Value)')
plt.ylabel('PC2 (Engagement Frequency)')
plt.legend()
plt.tight_layout()
plt.savefig('plot_clusters_pca.png')
plt.show()
print('plot_clusters_pca.png saved.')